# 🧠 Shopify AI Concierge: Product RAG Initialization

This notebook demonstrates how to connect to the local Prisma database and Shopify GraphQL Client to fetch products and variants for downstream Vector Database (FAISS/Pinecone) embedding.

### Prerequisites
Make sure you have installed the research requirements:
```bash
pip install -r requirements-research.txt
```

In [ ]:
import sys
import os
import asyncio
import pandas as pd

# Add the parent directory to sys.path to import from app/
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from app.core.database import prisma
from app.services.shopify.connection import get_active_shop_connection

### 1. Connect to Database and Load Shopify Client

We will load the first active store from the database to get its decrypted access tokens.

In [ ]:
async def get_shop_client():
    await prisma.connect()
    try:
        # Get the first active store
        store = await prisma.store.find_first(where={"is_active": True})
        if not store:
            raise Exception("No active store found in the database.")
        
        print(f"Found store: {store.shopify_domain}")
        
        # Get connection (assumes you have a way to bypass user_id check in research, 
        # or pass the store.userId)
        client = await get_active_shop_connection(user_id=store.userId, store_id=store.id)
        return client
    finally:
        await prisma.disconnect()

# Run the async function in the notebook event loop
client = await get_shop_client()
print("Shopify Client Ready!")

### 2. Fetch Products via Admin API

We use the Admin API here instead of the Storefront API because it provides more comprehensive data for indexing (like internal tags, standard product types, and inventory counts).

In [ ]:
async def fetch_all_products(client):
    gql = """
    query getAllProducts($first: Int!, $after: String) {
      products(first: $first, after: $after) {
        pageInfo {
          hasNextPage
          endCursor
        }
        edges {
          node {
            id
            title
            description
            productType
            vendor
            tags
            status
            featuredImage { url }
            variants(first: 50) {
              edges {
                node {
                  id
                  title
                  sku
                  price
                  inventoryQuantity
                  selectedOptions { name value }
                }
              }
            }
          }
        }
      }
    }
    """
    
    all_products = []
    has_next_page = True
    cursor = None
    
    while has_next_page:
        variables = {"first": 50, "after": cursor}
        data = await client.execute_admin(gql, variables)
        
        products_data = data.get("products", {})
        edges = products_data.get("edges", [])
        
        for edge in edges:
            all_products.append(edge["node"])
            
        page_info = products_data.get("pageInfo", {})
        has_next_page = page_info.get("hasNextPage", False)
        cursor = page_info.get("endCursor")
        
        print(f"Fetched {len(all_products)} products so far...")
        
    return all_products

products = await fetch_all_products(client)
print(f"\nTotal Products Fetched: {len(products)}")

### 3. Flatten and Prepare Data for Embedding

To make good semantic vectors, we need rich text. We'll combine the product title, description, vendor, type, and tags.

In [ ]:
def prepare_product_documents(products):
    docs = []
    for p in products:
        # Clean ID
        p_id = p.get("id", "").split("/")[-1]
        
        # Create a rich text representation
        title = p.get("title", "")
        desc = p.get("description", "")
        ptype = p.get("productType", "")
        vendor = p.get("vendor", "")
        tags = ", ".join(p.get("tags", []))
        
        # This is the string we will actually embed into the Vector DB
        embedding_text = f"Product: {title}.\nType: {ptype}.\nVendor: {vendor}.\nTags: {tags}.\nDescription: {desc}"
        
        docs.append({
            "id": p_id,
            "title": title,
            "embedding_text": embedding_text.strip(),
            "image_url": p.get("featuredImage", {}).get("url") if p.get("featuredImage") else None,
            "metadata": {
                "vendor": vendor,
                "product_type": ptype,
                "status": p.get("status")
            }
        })
    return pd.DataFrame(docs)

df_products = prepare_product_documents(products)
df_products.head()

In [ ]:
# Save to CSV for the next notebook
df_products.to_csv("shopify_products_prep.csv", index=False)
print("Saved prepared products to shopify_products_prep.csv")